# Скачивание документов тендеров с МАРКЕРА

API у меня нет (нужна отдельная подписка), поэтому скачиваю документы прямо через браузер с помощью Playwright.

Что делает скрипт:
1. Открывает браузер и логинится в МАРКЕР
2. Заходит на страницу каждой закупки и открывает вкладку **Документация**
3. Скачивает файлы в папку `docs/<registry_number>__<purchase_code>/raw/`
4. ZIP-архивы распаковывает автоматически, RAR — если есть unar/7z


In [ ]:
import re
import shutil
import subprocess
from pathlib import Path
from urllib.parse import urlparse
from playwright.async_api import async_playwright, TimeoutError as PlaywrightTimeoutError

import pandas as pd

ROOT = Path("/Users/arinazajceva/Desktop/диплом")
DATA = ROOT / "data_processed"
DOCS = ROOT / "docs"

RESULTS_MAIN = DATA / "marker_results_main_clean.csv"

print("results_main exists:", RESULTS_MAIN.exists())
print("docs dir:", DOCS)

results_main exists: True
docs dir: /Users/arinazajceva/Desktop/диплом/docs


Ключевые слова для фильтрации нужных документов (ТЗ, НМЦК, экспертиза и т.д.)

In [ ]:
KEYWORDS_PRIMARY = [
    "техническое задание",
    "тз",
    "задание на проектирование",
    "описание объекта",
    "обоснование нмцк",
    "нмцк",
    "экспертиз",
    "проект контракта",
    "контракт",
]

KEYWORDS_PSD = [
    "заключение экспертизы",
    "государственной экспертизы",
    "госэкспертиз",
    "экспертиз",
    "пояснительная записка",
    "поясн",
    "пз",
    "тэп",
    "технико-эконом",
    "проектная документац",
    "псд",
    "проект",
    "пд",
    "приложение № 1",
    "приложение 1",
]


def tender_dir(registry_number, purchase_code):
    return DOCS / f"{registry_number}__{purchase_code}"


def safe_name(name):
    name = (name or "").strip().replace("\u00a0", " ")
    name = re.sub(r"[\\/:*?\"<>|]", "_", name)
    name = re.sub(r"\s+", " ", name)
    return name[:180].strip() or "file"


def extract_zip(path, out_dir):
    import zipfile
    out_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(path) as z:
        z.extractall(out_dir)


def extract_rar(path, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    for cmd in ["unrar", "rar", "7z"]:
        if shutil.which(cmd):
            if cmd == "7z":
                r = subprocess.run(["7z", "x", f"-o{out_dir}", "-y", str(path)], capture_output=True)
            else:
                r = subprocess.run([cmd, "x", "-o+", "-y", str(path), str(out_dir)], capture_output=True)
            return r.returncode == 0
    return False


def pick_by_keywords(names):
    kw = [k.lower() for k in KEYWORDS_PRIMARY]
    return [i for i, n in enumerate(names) if any(k in (n or "").lower() for k in kw)]


def pick_psd_by_keywords(names):
    kw = [k.lower() for k in KEYWORDS_PSD]
    return [i for i, n in enumerate(names) if any(k in (n or "").lower() for k in kw)]


Если документы уже скачаны, но архивы не распаковались — можно распаковать отдельно:

In [ ]:
import os

# повторная распаковка архивов, если что-то не распаковалось при скачивании
def rerun_extract_all(docs_root=DOCS):
    os.environ["PATH"] = "/opt/homebrew/bin:/usr/local/bin:" + os.environ.get("PATH", "")

    rows = []
    for td in sorted(p for p in docs_root.iterdir() if p.is_dir()):
        raw_dir = td / "raw"
        ext_dir = td / "extracted"
        if not raw_dir.exists():
            continue
        ext_dir.mkdir(parents=True, exist_ok=True)

        files_before = sum(1 for p in ext_dir.rglob("*") if p.is_file())

        for z in sorted(raw_dir.rglob("*.zip")):
            extract_zip(z, ext_dir)

        for r in list(sorted(raw_dir.rglob("*.rar"))) + list(sorted(ext_dir.rglob("*.rar"))):
            extract_rar(r, ext_dir)

        files_after = sum(1 for p in ext_dir.rglob("*") if p.is_file())
        rows.append({"tender_dir": td.name, "files_before": files_before, "files_after": files_after})

    return pd.DataFrame(rows)


Смотрю, сколько файлов скачалось по каждому тендеру — проверяю, что все нормально

In [ ]:
tender_dirs = [p for p in DOCS.iterdir() if p.is_dir()]

def count_suffix(root, suffixes):
    c = {s: 0 for s in suffixes}
    if not root.exists():
        return c
    for p in root.rglob("*"):
        if not p.is_file():
            continue
        suf = p.suffix.lower()
        if suf in c:
            c[suf] += 1
    return c

rows = []
for td in sorted(tender_dirs):
    ext_dir = td / "extracted"
    raw_dir = td / "raw"

    ext = count_suffix(ext_dir, {".pdf", ".docx", ".doc", ".txt"})
    raw = count_suffix(raw_dir, {".pdf", ".docx", ".doc", ".zip", ".rar"})

    rows.append({
        "tender_dir": td.name,
        "extracted_pdf": ext[".pdf"],
        "extracted_docx": ext[".docx"],
        "extracted_doc": ext[".doc"],
        "extracted_txt": ext[".txt"],
        "raw_pdf": raw[".pdf"],
        "raw_docx": raw[".docx"],
        "raw_doc": raw[".doc"],
        "raw_zip": raw[".zip"],
        "raw_rar": raw[".rar"],
    })

print("tenders total:", len(rows))

empty = []
for r in rows:
    if (r["extracted_pdf"] + r["extracted_docx"] + r["extracted_doc"]) == 0:
        empty.append(r)

print("empty extracted (no pdf/doc/docx):", len(empty))

empty.sort(
    key=lambda r: (r["raw_rar"], r["raw_zip"], r["raw_pdf"], r["raw_docx"], r["raw_doc"]),
    reverse=True,
)

print("\nTop 30 empty extracted:")
for r in empty[:30]:
    print(
        r["tender_dir"],
        "| raw rar/zip/pdf/docx/doc:",
        r["raw_rar"], r["raw_zip"], r["raw_pdf"], r["raw_docx"], r["raw_doc"],
        "| extracted pdf/docx/doc/txt:",
        r["extracted_pdf"], r["extracted_docx"], r["extracted_doc"], r["extracted_txt"],
    )

tenders total: 160
empty extracted (no pdf/doc/docx): 47

Top 30 empty extracted:
160600005023000052__23-36452148490645201001-0039-002-0000-414 | raw rar/zip/pdf/docx/doc: 1 0 38 4 5 | extracted pdf/docx/doc/txt: 0 0 0 0
160300014425000009__25-36444003861644401001-0011-001-4120-414 | raw rar/zip/pdf/docx/doc: 0 16 0 0 0 | extracted pdf/docx/doc/txt: 0 0 0 0
108500000425004653__25-31215128193121501001-0304-001-0000-414 | raw rar/zip/pdf/docx/doc: 0 10 2 2 6 | extracted pdf/docx/doc/txt: 0 0 0 0
311200014624000057__24-21654006250165501001-0088-001-4120-414 | raw rar/zip/pdf/docx/doc: 0 10 0 4 2 | extracted pdf/docx/doc/txt: 0 0 0 0
311200014625000005__25-21654006250165501001-0029-001-4120-414 | raw rar/zip/pdf/docx/doc: 0 9 0 4 2 | extracted pdf/docx/doc/txt: 0 0 0 0
311200014625000010__25-21654006250165501001-0023-001-4120-414 | raw rar/zip/pdf/docx/doc: 0 9 0 4 2 | extracted pdf/docx/doc/txt: 0 0 0 0
311200014625000011__25-21654006250165501001-0022-001-4120-414 | raw rar/zip/pdf/docx/d

In [ ]:
import os, shutil

os.environ["PATH"] = "/opt/homebrew/bin:/usr/local/bin:" + os.environ.get("PATH", "")

print("rar:", shutil.which("rar"))
print("unrar:", shutil.which("unrar"))

rar: /opt/homebrew/bin/rar
unrar: /opt/homebrew/bin/unrar


Основная функция скачивания — открывает страницу закупки, находит файлы и сохраняет их, архивы (zip/rar) сразу распаковываются

In [ ]:
async def download_docs_for_lot(page, tender_root, mode="primary_then_all", include_psd_archives=False):
    from playwright.async_api import TimeoutError as PlaywrightTimeoutError

    raw_dir = tender_root / "raw"
    ext_dir = tender_root / "extracted"
    raw_dir.mkdir(parents=True, exist_ok=True)
    ext_dir.mkdir(parents=True, exist_ok=True)

    na_reason = None
    downloaded = 0
    rar_found = False
    rar_extracted = False

    # пробуем открыть вкладку "Документация"
    try:
        await page.get_by_text("Документация", exact=False).first.click(timeout=5000)
    except Exception:
        pass

    await page.wait_for_timeout(1500)

    def pick_filename(text):
        t = (text or "").strip()
        m = re.search(r"\[([^\]]+\.(?:pdf|docx?|zip|rar|7z))\]", t, flags=re.I)
        if m:
            return m.group(1)
        m = re.search(r"([^\s\\/]+\.(?:pdf|docx?|zip|rar|7z))", t, flags=re.I)
        if m:
            return m.group(1)
        lines = [x.strip() for x in t.splitlines() if x.strip()]
        return (lines[-1] if lines else "file")

    def save_and_extract(out):
        nonlocal rar_found, rar_extracted, na_reason
        suf = out.suffix.lower()
        if suf == ".zip":
            extract_zip(out, ext_dir)
        elif suf == ".rar":
            rar_found = True
            if extract_rar(out, ext_dir):
                rar_extracted = True
            else:
                na_reason = na_reason or "rar_needs_manual_extract"

    # ищем таблицу с кнопками "Скачать"
    rows = page.locator("table tr").filter(has=page.get_by_text("Скачать", exact=False))
    row_cnt = await rows.count()

    if row_cnt > 0:
        names = []
        statuses = []

        for i in range(row_cnt):
            r = rows.nth(i)
            cell_text = ((await r.locator("td").first.inner_text()) or "").strip()
            names.append(pick_filename(cell_text))
            tds = r.locator("td")
            td_cnt = await tds.count()
            st = ((await tds.nth(2).inner_text()) or "").strip() if td_cnt >= 3 else ""
            statuses.append(st)

        statuses_norm = [s.strip().lower() for s in statuses]

        # скачиваем только документы со статусом "Успешно"
        order = [i for i in range(row_cnt) if statuses_norm[i] == "успешно"]

        # если нужна проектная документация — добавляем по ключевым словам
        if include_psd_archives:
            for i in pick_psd_by_keywords(names):
                if i not in order and statuses_norm[i] != "неизвестно":
                    order.append(i)

        n = 1
        for i in order:
            r = rows.nth(i)
            name = safe_name(names[i])
            out = raw_dir / f"{n:02d}__{name}"
            n += 1

            try:
                async with page.expect_download(timeout=60000) as dl_info:
                    await r.get_by_text("Скачать", exact=False).first.click(timeout=10000)
                dl = await dl_info.value
                await dl.save_as(str(out))
                downloaded += 1
                save_and_extract(out)
            except Exception as e:
                na_reason = na_reason or type(e).__name__
                continue

        # распаковываем вложенные RAR (могли появиться внутри ZIP)
        for rar in sorted(ext_dir.rglob("*.rar")):
            rar_found = True
            if extract_rar(rar, ext_dir):
                rar_extracted = True
            else:
                na_reason = na_reason or "rar_needs_manual_extract"

        if downloaded == 0 and na_reason is None:
            na_reason = "no_links_found"

        return {
            "ok": downloaded > 0,
            "downloaded": downloaded,
            "rar_found": rar_found,
            "rar_extracted": rar_extracted,
            "na_reason": na_reason,
        }

    # запасной путь: ищем прямые ссылки на файлы (если таблицы нет)
    links = page.locator("a[href]")
    cnt = await links.count()

    hrefs = []
    names = []

    for i in range(cnt):
        a = links.nth(i)
        href = (await a.get_attribute("href")) or ""
        txt = ((await a.inner_text()) or "").strip()
        if not href:
            continue
        if any(ext in href.lower() for ext in [".pdf", ".doc", ".docx", ".zip", ".rar"]):
            hrefs.append(href)
            names.append(txt or Path(urlparse(href).path).name)

    if not hrefs:
        return {"ok": False, "downloaded": 0, "na_reason": "no_links_found"}

    picked = pick_by_keywords(names)
    order = picked if (mode == "primary_then_all" and picked) else list(range(len(hrefs)))

    n = 1
    for idx in order:
        href = hrefs[idx]
        name = safe_name(names[idx] or Path(urlparse(href).path).name)

        if href.startswith("/"):
            base = page.url.split("/Card/")[0] if "/Card/" in page.url else page.url.rstrip("/")
            href = f"{base}{href}"

        out = raw_dir / f"{n:02d}__{name}"
        n += 1

        try:
            async with page.expect_download(timeout=60000) as dl_info:
                await page.evaluate("(u) => window.open(u, '_blank')", href)
            dl = await dl_info.value
            await dl.save_as(str(out))
            downloaded += 1
            save_and_extract(out)
        except Exception:
            continue

    for rar in sorted(ext_dir.rglob("*.rar")):
        rar_found = True
        if extract_rar(rar, ext_dir):
            rar_extracted = True

    return {
        "ok": downloaded > 0,
        "downloaded": downloaded,
        "rar_found": rar_found,
        "rar_extracted": rar_extracted,
        "na_reason": na_reason,
    }


Загружаю список закупок и запускаю скачивание

In [94]:
results_main = pd.read_csv(RESULTS_MAIN)
work = results_main.loc[results_main["publication_url"].notna()].copy()
print('rows:', len(work))
work[["registry_number", "purchase_code", "publication_url"]].head()


rows: 157


,registry_number,purchase_code,publication_url
0,6448182,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...
1,8815987,NaN,https://analytics.marker-zakupki.ru/Card/Lot/1...
2,101500000322000125,22-30248005212024801001-0031-001-4120-414,https://analytics.marker-zakupki.ru/Card/Lot/1...
3,101500000322000183,22-20278176470027601001-0348-001-4120-414,https://analytics.marker-zakupki.ru/Card/Lot/1...
4,101500000322000257,22-20278176470027601001-0348-002-4120-414,https://analytics.marker-zakupki.ru/Card/Lot/1...


Пробный прогон:

In [ ]:
N = 15
sample = work[work["registry_number"].astype(str) == "101500000322000257"].head(1)

async def run_sample():
    log_rows = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False, channel="chrome")
        ctx = await browser.new_context(accept_downloads=True, ignore_https_errors=True)
        page = await ctx.new_page()

        await page.goto("https://analytics.marker-zakupki.ru/", wait_until="domcontentloaded", timeout=60000)
        print("Открой браузер, залогинься на маркере, потом нажми Enter здесь")
        input("Enter после логина...")

        for _, row in sample.iterrows():
            reg = str(row.get("registry_number"))
            pc = str(row.get("purchase_code") or "")
            url = str(row.get("publication_url"))

            tender_root = tender_dir(reg, pc)

            await page.goto(url, wait_until="domcontentloaded")
            await page.wait_for_timeout(600)

            res = await download_docs_for_lot(page, tender_root, mode="primary_then_all")
            res.update({
                "registry_number": reg,
                "purchase_code": pc,
                "publication_url": url,
                "docs_dir": str(tender_root),
            })
            log_rows.append(res)

        await browser.close()

    return pd.DataFrame(log_rows)

await run_sample()


Сейчас залогинься в открывшемся браузере. Потом вернись сюда и нажми Enter в консоли ноутбука.


,ok,downloaded,rar_found,rar_extracted,na_reason,registry_number,purchase_code,publication_url,docs_dir
0,True,33,True,False,rar_needs_manual_extract,101500000322000257,22-20278176470027601001-0348-002-4120-414,https://analytics.marker-zakupki.ru/Card/Lot/1...,/Users/arinazajceva/Desktop/диплом/docs/101500...


Полный прогон:

In [ ]:
OUT = DATA / "marker_docs_download_log.csv"

async def run_all():
    log_rows = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False, channel="chrome")
        ctx = await browser.new_context(accept_downloads=True, ignore_https_errors=True)
        page = await ctx.new_page()

        await page.goto("https://analytics.marker-zakupki.ru/", wait_until="domcontentloaded", timeout=60000)
        print("Залогинься в браузере. Потом нажми Enter здесь.")
        input("Enter после логина...")

        for _, row in work.iterrows():
            reg = str(row.get("registry_number"))
            pc = str(row.get("purchase_code") or "")
            url = str(row.get("publication_url"))

            tender_root = tender_dir(reg, pc)

            await page.goto(url, wait_until="domcontentloaded")
            await page.wait_for_timeout(600)

            res = await download_docs_for_lot(page, tender_root, mode="primary_then_all")
            res.update({
                "registry_number": reg,
                "purchase_code": pc,
                "publication_url": url,
                "docs_dir": str(tender_root),
            })
            log_rows.append(res)

        await browser.close()

    log_df = pd.DataFrame(log_rows)
    log_df.to_csv(OUT, index=False)
    print("сохранила:", OUT)
    return log_df

log_df = await run_all()
log_df.head()


Сейчас залогинься в открывшемся браузере. Потом вернись сюда и нажми Enter в консоли ноутбука.
saved: /Users/arinazajceva/Desktop/диплом/data_processed/marker_docs_download_log.csv


,ok,downloaded,na_reason,registry_number,purchase_code,publication_url,docs_dir,rar_found,rar_extracted
0,False,0,no_links_found,6448182,nan,https://analytics.marker-zakupki.ru/Card/Lot/1...,/Users/arinazajceva/Desktop/диплом/docs/644818...,NaN,NaN
1,False,0,no_links_found,8815987,nan,https://analytics.marker-zakupki.ru/Card/Lot/1...,/Users/arinazajceva/Desktop/диплом/docs/881598...,NaN,NaN
2,True,19,None,101500000322000125,22-30248005212024801001-0031-001-4120-414,https://analytics.marker-zakupki.ru/Card/Lot/1...,/Users/arinazajceva/Desktop/диплом/docs/101500...,True,True
3,True,11,None,101500000322000183,22-20278176470027601001-0348-001-4120-414,https://analytics.marker-zakupki.ru/Card/Lot/1...,/Users/arinazajceva/Desktop/диплом/docs/101500...,False,False
4,True,33,rar_needs_manual_extract,101500000322000257,22-20278176470027601001-0348-002-4120-414,https://analytics.marker-zakupki.ru/Card/Lot/1...,/Users/arinazajceva/Desktop/диплом/docs/101500...,True,False


In [17]:
log_df.shape

(157, 9)

Докачка для закупок, где не нашлась площадь — пробую скачать документацию проекта еще раз

In [ ]:
MISSING_ALL = DATA / "marker_docs_area_missing_all.csv"
MISSING_TOP30 = DATA / "marker_docs_area_missing_top30.csv"

MISSING = MISSING_ALL if MISSING_ALL.exists() else MISSING_TOP30
print("missing list file:", MISSING)
print("missing list exists:", MISSING.exists())

missing_df = pd.read_csv(MISSING) if MISSING.exists() else pd.DataFrame()
missing_df["purchase_code"] = missing_df["purchase_code"].fillna("")

need_keys = set(
    (str(r), str(pc)) for r, pc in zip(missing_df.get("registry_number", []), missing_df.get("purchase_code", []))
)

results_main = pd.read_csv(RESULTS_MAIN)
results_main["purchase_code"] = results_main["purchase_code"].fillna("")

work_missing = results_main[
    results_main.apply(lambda x: (str(x.get("registry_number")), str(x.get("purchase_code"))) in need_keys, axis=1)
].copy()
work_missing = work_missing.loc[work_missing["publication_url"].notna()].copy()

print("tenders to extra-download:", len(work_missing))

missing list file: /Users/arinazajceva/Desktop/диплом/data_processed/marker_docs_area_missing_all.csv
missing list exists: True
tenders to extra-download: 26


In [ ]:
async def run_extra_psd_download(work_df):
    log_rows = []

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=False, channel="chrome")
        ctx = await browser.new_context(accept_downloads=True, ignore_https_errors=True)
        page = await ctx.new_page()

        await page.goto("https://analytics.marker-zakupki.ru/", wait_until="domcontentloaded", timeout=60000)
        print("Залогинься в браузере. Потом нажми Enter.")
        input("Enter после логина...")

        for _, row in work_df.iterrows():
            reg = str(row.get("registry_number"))
            pc = str(row.get("purchase_code") or "")
            url = str(row.get("publication_url"))

            tender_root = tender_dir(reg, pc)

            await page.goto(url, wait_until="domcontentloaded")
            await page.wait_for_timeout(600)

            res = await download_docs_for_lot(
                page,
                tender_root,
                mode="primary_then_all",
                include_psd_archives=True,
            )
            res.update({
                "registry_number": reg,
                "purchase_code": pc,
                "publication_url": url,
                "docs_dir": str(tender_root),
                "extra_mode": "psd_by_keywords",
            })
            log_rows.append(res)

        await browser.close()

    return pd.DataFrame(log_rows)

In [96]:
extra_log = await run_extra_psd_download(work_missing)
extra_log.to_csv(DATA / "marker_docs_download_log_extra_psd.csv", index=False)
extra_log.head()


Залогинься в браузере. Потом вернись сюда и нажми Enter.


,ok,downloaded,rar_found,rar_extracted,na_reason,registry_number,purchase_code,publication_url,docs_dir,extra_mode
0,True,11,False,False,None,101500000322000183,22-20278176470027601001-0348-001-4120-414,https://analytics.marker-zakupki.ru/Card/Lot/1...,/Users/arinazajceva/Desktop/диплом/docs/101500...,psd_by_keywords
1,True,5,False,False,None,101500000322000403,22-20278176470027601001-0859-001-4120-414,https://analytics.marker-zakupki.ru/Card/Lot/1...,/Users/arinazajceva/Desktop/диплом/docs/101500...,psd_by_keywords
2,True,6,False,False,None,101500000322000444,22-20278176470027601001-0866-001-4120-414,https://analytics.marker-zakupki.ru/Card/Lot/1...,/Users/arinazajceva/Desktop/диплом/docs/101500...,psd_by_keywords
3,True,5,False,False,None,108500000425000132,25-31215128193121501001-0041-001-4120-414,https://analytics.marker-zakupki.ru/Card/Lot/1...,/Users/arinazajceva/Desktop/диплом/docs/108500...,psd_by_keywords
4,True,8,False,False,None,108500000425004653,25-31215128193121501001-0304-001-0000-414,https://analytics.marker-zakupki.ru/Card/Lot/1...,/Users/arinazajceva/Desktop/диплом/docs/108500...,psd_by_keywords
